In [ ]:
Option = 0 #Option=0 -> discrete, Option=1 -> continuous

In [ ]:
import struct
import numpy as np

def load_mnist_images(filename,image_file):
    with open(filename, 'rb') as f:
        if image_file: #影像檔
          # 讀取 header
          magic, num_images, rows, cols = struct.unpack('>IIII', f.read(16))
          #前4個分別為magic, num_images, rows, cols (pdf上的unsigned byte從17開始)
          # 讀取影像資料
          image_data = f.read(num_images * rows * cols)
          images = np.frombuffer(image_data, dtype=np.uint8) #Buffer -> Numpy Array
          images = images.reshape((num_images, rows, cols)) #確保shape
          return images

        else: #標籤檔
          # 讀取 header
          magic, num_labels = struct.unpack('>II', f.read(8))
          #前2個分別為magic, num_images (pdf上的unsigned byte從9開始)
          label_data = f.read(num_labels)
          labels = np.frombuffer(label_data, dtype=np.uint8) #Buffer -> Numpy Array
          return labels

train_images = load_mnist_images('train-images.idx3-ubyte_',True)
train_labels = load_mnist_images('train-labels.idx1-ubyte_',False)
test_images = load_mnist_images('t10k-images.idx3-ubyte_',True)
test_labels = load_mnist_images('t10k-labels.idx1-ubyte_',False)

In [ ]:
print(train_images.shape) #60000張圖, 每張Features: 28x28 (X)
print(train_labels.shape) #60000個答案 (Y)
print(test_images.shape) #10000張圖, 每張Features: 28x28 (X)
print(test_labels.shape) #10000個答案 (Y)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [ ]:
def imagination_discrete(Prob_Pic):
    print("Imagination of numbers in Bayesian classifier:")
    for digit in range(10): #印出0-9的統計後pattern
        print(f"{digit}:")
        pattern = np.argmax(Prob_Pic[digit], axis=1).reshape(28,28)
        for row in pattern: #pattern中每橫排 (逐排打印推測)
            print(" ".join(str(int(val > np.median(row))) for val in row))
            #若pixel val > 該排中位數 -> 1; 反之為0

def imagination_continuous(means):
    print("Imagination of numbers in Bayesian classifier:")
    for digit in range(10): #印出0-9的統計後pattern
        print(f"{digit}:")
        pattern = means[digit].reshape(28,28)
        for row in pattern: #pattern中每橫排
            print(" ".join(str(int(val > 128)) for val in row))  # 128 作為 threshold (0-255一半) #若過半則設1

In [ ]:
def pixel_to_bin(pixel):
  return pixel//32 #分成32bins

def Prob_09_produced(labels):
  #每個數都是0-9
  Prob_09 = np.zeros(10)
  for i in range(len(labels)):
    Prob_09[labels[i]]+=1 #Prob_09紀錄了0 - 9各自機率
  Prob_09 = Prob_09/len(labels) #Prob_09 -> P(Y)
  return Prob_09

def Naive_Bayes(train_images,train_labels):
  Prob_09 = Prob_09_produced(train_labels) #Prob_09 -> P(Y)

  Prob_Pic = np.zeros((10,784,32)) #P(X|Y), 10:0-9; 784:28x28; 32個bins

  Flatten_IMG = train_images.reshape(len(train_images), -1) #28x28 -> 784 pixels (i=0 - 783)
  bin_id = pixel_to_bin(Flatten_IMG) #256個bins改分成8個bins (0-31), (32-63), etc.
  for label in range(10): #先選定我們某圈要統計的label
    current_label_images = bin_id[train_labels == label] #在train_labels中尋找等於label的,若等於則存取對應圖片進current label images
    #歸屬當前label之圖片庫
    for flat_pixel in range(784):
      counts = np.bincount(current_label_images[:, flat_pixel], minlength=32) #統計此圖片庫中的每個圖的784個pixel表現
      #Prob_Pic[label][flat_pixel][:] = np.sum(current_label_images[:, flat_pixel] == np.arange(8)[:, None], axis=0)
      Prob_Pic[label, flat_pixel, :] = counts #Prob_Pic裝載某label下某個flat_pixel下的統計數量

  Prob_Pic += 1  # Laplace smoothing, 避免underflow
  Prob_Pic = Prob_Pic / Prob_Pic.sum(axis=2, keepdims=True)  # 每個像素位置的條件機率

  imagination_discrete(Prob_Pic) #將算出的pattern印出

  return Prob_Pic, Prob_09

def compute_posterior(img, Prob_Pic, Prob_09): #針對各張照片
  log_posteriors = np.zeros(10)
  for y in range(10):
      log_prob = np.log(Prob_09[y]) #In log scale to avoid underflow
      for i, pixel in enumerate(img.flatten()): #壓成(784,)
          bin_id = pixel_to_bin(pixel) #轉換進32個bins
          log_prob += np.log(Prob_Pic[y][i][bin_id]) #將各pixel機率加總
      log_posteriors[y] = log_prob #將加總機率當作"預測為此digit"之可能性

  max_log = max(log_posteriors)
  log_total = max_log + np.log(sum(np.exp(log_p - max_log) for log_p in log_posteriors)) #實際上這段含有/P(X) , Normalize.

  posterior_probs = np.exp(log_posteriors - log_total)  # shape: (10,)
  return posterior_probs

def Naive_Bayes_Continuous(train_images,train_labels):
  Prob_09 = Prob_09_produced(train_labels) #Prob_09 -> P(Y)

  means = np.zeros((10, 784)) #Pattern means of each pixel
  variances = np.zeros((10, 784)) #Pattern variances of each pixel

  Flatten_IMG = train_images.reshape(len(train_images), -1)
  for label in range(10): #在愈統計的label下, 計算出現的平均和variances (統計某一pixel下做成distribution)
    label_images = Flatten_IMG[train_labels == label] #和discrete相同先取出當前label的images.
    means[label] = np.mean(label_images, axis=0) #當前label下所有images下每個pixel的mean (假如有N張圖,每張784pixels, 則output大小為1x784)
    variances[label] = (np.std(label_images, axis=0))**2 + 1e-5 #var = std^2, 加入1e-5以防log(0) -> 否則可能印出全部nan.

  imagination_continuous(means) #將算出的pattern印出

  return means, variances, Prob_09

def compute_posterior_Continuous(img, means, variances, Prob_09): #針對各張照片
  img = img.flatten()
  log_posteriors = np.zeros(10)
  for y in range(10): #底下為gaussian distribution函數取log (Gauss: (1/np.sqrt(2*np.pi*np.std**2))*np.exp(-(x-np.mean)**2/2*np.std**2))
      log_prob = np.log(Prob_09[y])
      log_prob += -0.5 * np.sum(np.log(2 * np.pi * variances[y]))
      log_prob += -0.5 * np.sum(((img - means[y])**2) / variances[y])
      log_posteriors[y] = log_prob

  max_log = np.max(log_posteriors)
  log_total = max_log + np.log(np.sum(np.exp(log_posteriors - max_log))) #Normalized
  posterior_probs = np.exp(log_posteriors - log_total)
  return posterior_probs


In [ ]:
def Mode_Select_VER_Train(Option):
  Return_Pack = []
  if Option==0: #Discrete
    Prob_Pic, Prob_09 = Naive_Bayes(train_images,train_labels)
    Pred = np.zeros((train_labels.shape[0],10))
    Final_Decision = np.zeros(train_labels.shape[0])
    for i in range(train_labels.shape[0]):
      if i%1000==0: #每1000圈印出進度
        print("Training Process [",i," / ",train_labels.shape[0],"]")
      Pred[i] = compute_posterior(train_images[i], Prob_Pic, Prob_09) #計算此照片屬於各label之可能性
      Final_Decision[i] = np.argmax(Pred[i])
    Return_Pack.append(Prob_Pic)
    Return_Pack.append(Prob_09)
    #Return_Pack = [Prob_Pic, Prob_09]
  elif Option==1: #Continuous
    means, variances, Prob_09 = Naive_Bayes_Continuous(train_images,train_labels)
    Pred = np.zeros((train_labels.shape[0],10))
    Final_Decision = np.zeros(train_labels.shape[0])
    for i in range(train_labels.shape[0]):
      if i%1000==0: #每1000圈印出進度
        print("Training Process [",i," / ",train_labels.shape[0],"]")
      Pred[i] = compute_posterior_Continuous(train_images[i], means, variances, Prob_09) #計算此照片屬於各label之可能性
      Final_Decision[i] = np.argmax(Pred[i])
    Return_Pack.append(means)
    Return_Pack.append(variances)
    Return_Pack.append(Prob_09)
    #Return_Pack = [means, variances, Prob_09]
  else:
    print("Wrong Setting.")

  correct = np.sum(Final_Decision == train_labels) #若Final_Decision[i]==train_labels[i]則+1
  total = len(train_labels)
  error_rate = 1 - correct / total
  print(f"Training Error Rate: {error_rate:.4f}")
  return Return_Pack

In [ ]:
def Model_Select_VER_Test(Option, Return_Pack):
  Test_Pred = np.zeros((test_labels.shape[0], 10)) #對應10個digits的機率
  Final_Test = np.zeros(test_labels.shape[0]) #Test Pred最終選擇(最大者)
  for i in range(test_labels.shape[0]):
    if Option==0: #Discrete
      Test_Pred[i] = compute_posterior(test_images[i], Return_Pack[0], Return_Pack[1])
    elif Option==1: #Continuous
      Test_Pred[i] = compute_posterior_Continuous(test_images[i], Return_Pack[0], Return_Pack[1], Return_Pack[2])
    Final_Test[i] = np.argmax(Test_Pred[i])

    if i % 1000 == 0: #每1000圈印出posterior
      print("Testing Process [",i," / ",test_labels.shape[0],"]")
      print(f"Posterior (log scale), sample {i}:")
      for j, p in enumerate(Test_Pred[i]):
          print(f" {j}: {p:.8f}")
      print(f" Prediction: {Final_Test[i]}, Ans: {test_labels[i]}\n")

  correct_test = np.sum(Final_Test == test_labels)
  error_rate_test = 1 - correct_test / len(test_labels)
  print(f"Test Error Rate: {error_rate_test:.4f}")

In [ ]:
Return_Pack = Mode_Select_VER_Train(Option)
Model_Select_VER_Test(Option, Return_Pack)

Imagination of numbers in Bayesian classifier:
0:
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 0 0 0 1 1 1 1 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0
0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0
0 0 0 0 0 0 0 1 1 1 1 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0
0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0
0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0
0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 